<a href="https://colab.research.google.com/github/sredhalucca-hash/airline-passengers/blob/main/MatchWise_Cricket_ML_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install xgboost shap joblib


## 2. Environment setup

In [ ]:
# Run this cell in Google Colab.

import os
import random
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from google.colab import files

import sklearn
from sklearn import set_config
set_config(transform_output="pandas")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Python-ready environment")
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("GPU check skipped:", exc)

Python-ready environment
Pandas: 2.2.2
Scikit-learn: 1.6.1
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## 3. Upload and load the dataset

In [ ]:
# Upload the CSV file when prompted.
uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith(".csv")]

if not csv_files:
    raise FileNotFoundError("Please upload the MatchWise CSV dataset.")

DATA_PATH = csv_files[0]
df = pd.read_csv(DATA_PATH)

print(f"Loaded: {DATA_PATH}")
print("Shape:", df.shape)
display(df.head())

KeyboardInterrupt: 

In [ ]:
# Initial inspection
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.rename("dtype").to_frame())

print("\nSummary:")
display(df.describe(include="all").T)

print("\nDuplicate rows:", df.duplicated().sum())
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).rename("missing_count").to_frame())

### Dataset observations

The uploaded data has no fully duplicated rows. Several result-related columns contain many missing values because different winning methods apply to different matches. `event_name`, `ground_city`, and `player_of_the_match` also contain missing values.

Missingness is not automatically an error. It must be interpreted in context. For example, a match won by wickets will normally have no `margin_runs`.

## 4. Data cleaning and feature engineering

In [ ]:
# Standardize column names and text values.
df.columns = (
    df.columns.str.strip()
              .str.lower()
              .str.replace(" ", "_")
)

object_columns = df.select_dtypes(include="object").columns
for col in object_columns:
    df[col] = df[col].astype("string").str.strip()

# Parse date and derive time-based features.
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.dayofweek

# Remove exact duplicates if any appear after future dataset updates.
df = df.drop_duplicates().reset_index(drop=True)

# Binary classification target.
df["team_1_win"] = (df["winner"] == df["team_1"]).astype(int)

print("Cleaned shape:", df.shape)
print("Date parsing failures:", df["date"].isna().sum())
print("Target distribution:")
display(df["team_1_win"].value_counts().rename("count").to_frame())
display((df["team_1_win"].value_counts(normalize=True) * 100).round(2).rename("percentage").to_frame())

## 5. Data-leakage audit

A leakage feature contains information that would not be known at prediction time. Using it can produce impressive but dishonest scores.

For pre-match winner prediction:

| Field | Keep? | Reason |
|---|---:|---|
| Teams, venue, event, toss | Yes | Known before play |
| Date-derived fields | Yes | Known before play |
| Final scores | No | Known only after play |
| Winning margin | No | Directly reveals result |
| Player of the match | No | Awarded after match |
| Winner | No | This is the answer |
| Match IDs | No | Identifier with little causal meaning |

In [ ]:
classification_features = [
    "event_name",
    "ground_name",
    "ground_city",
    "team_1",
    "team_2",
    "toss_winner",
    "toss_decision",
    "year",
    "month",
    "day_of_week",
]

leakage_columns = [
    "winner",
    "team_1_total_runs",
    "team_2_total_runs",
    "margin_runs",
    "margin_wickets",
    "winning_method",
    "player_of_the_match",
]

identifier_columns = ["match_number", "match_id", "date"]

X = df[classification_features].copy()
y = df["team_1_win"].copy()

print("Classification features:", classification_features)
print("Excluded leakage columns:", leakage_columns)
print("Target:", y.name)

## 6. Exploratory data analysis

In [ ]:
# Target balance
ax = y.value_counts().sort_index().plot(kind="bar")
ax.set_title("Team 1 Win Distribution")
ax.set_xlabel("Target: 0 = Team 1 did not win, 1 = Team 1 won")
ax.set_ylabel("Number of matches")
plt.show()

# Toss decision
pd.crosstab(df["toss_decision"], df["team_1_win"], normalize="index").plot(kind="bar")
plt.title("Team 1 Win Rate by Toss Decision")
plt.ylabel("Proportion")
plt.show()

# Matches by year
df["year"].value_counts().sort_index().plot()
plt.title("Number of Matches by Year")
plt.xlabel("Year")
plt.ylabel("Matches")
plt.show()

# Most frequent teams
team_counts = pd.concat([df["team_1"], df["team_2"]]).value_counts().head(15)
team_counts.sort_values().plot(kind="barh")
plt.title("15 Most Frequently Observed Teams")
plt.xlabel("Match appearances")
plt.show()

### Interpretation prompts
- Is the binary target severely imbalanced?
- Does toss decision appear associated with match outcome?
- Are some teams represented far more often than others?
- Why might a random split make historical sports prediction look easier than future prediction?

## 7. Time-aware train, validation, and test split

In [ ]:
# Sort chronologically so the model learns from the past and is tested on later matches.
ordered = df.sort_values("date").reset_index(drop=True)

n = len(ordered)
train_end = int(n * 0.70)
valid_end = int(n * 0.85)

train_df = ordered.iloc[:train_end].copy()
valid_df = ordered.iloc[train_end:valid_end].copy()
test_df  = ordered.iloc[valid_end:].copy()

X_train = train_df[classification_features]
y_train = train_df["team_1_win"]

X_valid = valid_df[classification_features]
y_valid = valid_df["team_1_win"]

X_test = test_df[classification_features]
y_test = test_df["team_1_win"]

print("Train:", X_train.shape, train_df["date"].min(), "to", train_df["date"].max())
print("Valid:", X_valid.shape, valid_df["date"].min(), "to", valid_df["date"].max())
print("Test :", X_test.shape, test_df["date"].min(), "to", test_df["date"].max())

A chronological split is more realistic than a random split because the practical question is: **Can patterns learned from earlier matches generalize to later matches?**

The test set remains untouched until final model selection.

## 8. Leakage-safe preprocessing pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

categorical_features = [
    "event_name", "ground_name", "ground_city",
    "team_1", "team_2", "toss_winner", "toss_decision"
]
numeric_features = ["year", "month", "day_of_week"]

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=2)),
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, categorical_features),
    ("numeric", numeric_pipeline, numeric_features),
], remainder="drop")

preprocessor

`Pipeline` and `ColumnTransformer` ensure that imputation, encoding, and scaling are learned only from training data. This prevents subtle leakage during cross-validation.

## 9. Evaluation functions

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

results = []
trained_models = {}

def evaluate_classifier(name, pipeline, X_train, y_train, X_valid, y_valid, cv=None):
    start = time.perf_counter()
    pipeline.fit(X_train, y_train)
    fit_time = time.perf_counter() - start

    start = time.perf_counter()
    train_pred = pipeline.predict(X_train)
    valid_pred = pipeline.predict(X_valid)
    pred_time = time.perf_counter() - start

    train_prob = pipeline.predict_proba(X_train)[:, 1] if hasattr(pipeline, "predict_proba") else None
    valid_prob = pipeline.predict_proba(X_valid)[:, 1] if hasattr(pipeline, "predict_proba") else None

    train_f1 = f1_score(y_train, train_pred)
    valid_f1 = f1_score(y_valid, valid_pred)

    cv_mean = np.nan
    cv_std = np.nan
    if cv is not None:
        scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1)
        cv_mean, cv_std = scores.mean(), scores.std()
        print(name, "CV F1 scores:", np.round(scores, 4))

    row = {
        "Model": name,
        "Train F1": train_f1,
        "Validation F1": valid_f1,
        "CV Mean F1": cv_mean,
        "CV Std": cv_std,
        "Validation Accuracy": accuracy_score(y_valid, valid_pred),
        "Validation Precision": precision_score(y_valid, valid_pred, zero_division=0),
        "Validation Recall": recall_score(y_valid, valid_pred, zero_division=0),
        "Validation ROC-AUC": roc_auc_score(y_valid, valid_prob) if valid_prob is not None else np.nan,
        "Overfitting Gap": train_f1 - valid_f1,
        "Training Time (s)": fit_time,
        "Prediction Time (s)": pred_time,
    }
    results.append(row)
    trained_models[name] = pipeline
    return row

tscv = TimeSeriesSplit(n_splits=5)

## 10. Baseline models

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

# Redefine preprocessor and its components to handle pd.NA
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
import numpy as np

def convert_pd_na_to_np_nan(X):
    """Converts pandas.NA values in a DataFrame to numpy.nan."""
    # ColumnTransformer passes a DataFrame containing only the selected columns.
    # Explicitly convert to object dtype to ensure np.nan is correctly handled
    # for columns that might have StringDtype and pd.NA.
    X_converted = X.astype(object).replace({pd.NA: np.nan})
    return X_converted

categorical_features = [
    "event_name", "ground_name", "ground_city",
    "team_1", "team_2", "toss_winner", "toss_decision"
]
numeric_features = ["year", "month", "day_of_week"]

categorical_pipeline = Pipeline([
    ("to_np_nan", FunctionTransformer(convert_pd_na_to_np_nan, validate=False)),
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=2, sparse_output=False)),
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, categorical_features),
    ("numeric", numeric_pipeline, numeric_features),
], remainder="drop")
# End of preprocessor re-definition

dummy_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent")),
])

logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)),
])

evaluate_classifier(
    "Dummy Baseline", dummy_pipeline,
    X_train, y_train, X_valid, y_valid, cv=tscv
)

evaluate_classifier(
    "Logistic Regression", logistic_pipeline,
    X_train, y_train, X_valid, y_valid, cv=tscv
)

display(pd.DataFrame(results).sort_values("Validation F1", ascending=False))

A dummy model is the minimum benchmark. A useful machine-learning model should outperform it on meaningful metrics, not merely appear sophisticated.

## 11. Overfitting and regularization demonstration

In [ ]:
from sklearn.tree import DecisionTreeClassifier

deep_tree = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=SEED)),
])

regularized_tree = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=SEED
    )),
])

evaluate_classifier(
    "Unrestricted Decision Tree", deep_tree,
    X_train, y_train, X_valid, y_valid, cv=tscv
)

evaluate_classifier(
    "Regularized Decision Tree", regularized_tree,
    X_train, y_train, X_valid, y_valid, cv=tscv
)

display(pd.DataFrame(results).sort_values("Validation F1", ascending=False))

## 12. Learning curve

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, valid_scores = learning_curve(
    logistic_pipeline,
    X_train,
    y_train,
    cv=tscv,
    scoring="f1",
    train_sizes=np.linspace(0.2, 1.0, 5),
    n_jobs=-1,
)

plt.plot(train_sizes, train_scores.mean(axis=1), marker="o", label="Training F1")
plt.plot(train_sizes, valid_scores.mean(axis=1), marker="o", label="Cross-validation F1")
plt.xlabel("Training examples")
plt.ylabel("F1 score")
plt.title("Learning Curve: Logistic Regression")
plt.legend()
plt.show()

## 13. Hyperparameter tuning with RandomizedSearchCV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

tuning_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=3000, random_state=SEED)),
])

param_distributions = {
    "model__C": loguniform(1e-3, 1e2),
    "model__class_weight": [None, "balanced"],
    "model__solver": ["liblinear", "saga"],
}

search = RandomizedSearchCV(
    tuning_pipeline,
    param_distributions=param_distributions,
    n_iter=12,
    scoring="f1",
    cv=tscv,
    n_jobs=-1,
    random_state=SEED,
    verbose=1,
)

start = time.perf_counter()
search.fit(X_train, y_train)
print("Tuning time:", round(time.perf_counter() - start, 2), "seconds")
print("Best parameters:", search.best_params_)
print("Best CV F1:", round(search.best_score_, 4))

tuned_logistic = search.best_estimator_
evaluate_classifier(
    "Tuned Logistic Regression", tuned_logistic,
    X_train, y_train, X_valid, y_valid, cv=tscv
)

## 14. Ensemble models

In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier

random_forest = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=4,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=SEED
    )),
])

extra_trees = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ExtraTreesClassifier(
        n_estimators=300,
        max_depth=14,
        min_samples_leaf=3,
        class_weight="balanced",
        n_jobs=-1,
        random_state=SEED
    )),
])

evaluate_classifier("Random Forest", random_forest,
                    X_train, y_train, X_valid, y_valid, cv=tscv)

evaluate_classifier("Extra Trees", extra_trees,
                    X_train, y_train, X_valid, y_valid, cv=tscv)

display(pd.DataFrame(results).sort_values("Validation F1", ascending=False))

### Bagging versus boosting

- **Bagging** trains many models in parallel and combines their outputs. It is like asking several independent experts and taking a vote.
- **Boosting** trains models sequentially, with each new model focusing more strongly on earlier mistakes. It is like a coach reviewing errors after every practice round.

## 15. XGBoost with optional CUDA acceleration

In [ ]:
from xgboost import XGBClassifier

# XGBoost 2.x uses device="cuda" when a compatible GPU build is available.
# The code falls back to CPU if GPU training fails.
def build_xgb(device="cuda"):
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(
            n_estimators=350,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            device=device,
            random_state=SEED,
        )),
    ])

try:
    xgb_pipeline = build_xgb("cuda")
    xgb_pipeline.fit(X_train, y_train)
    print("XGBoost GPU training succeeded.")
except Exception as exc:
    print("GPU training unavailable; falling back to CPU.")
    print("Reason:", str(exc)[:300])
    xgb_pipeline = build_xgb("cpu")

evaluate_classifier(
    "XGBoost", xgb_pipeline,
    X_train, y_train, X_valid, y_valid, cv=tscv
)

## 16. Model comparison and provisional selection

In [ ]:
comparison = (
    pd.DataFrame(results)
      .drop_duplicates(subset="Model", keep="last")
      .sort_values(["Validation F1", "CV Mean F1"], ascending=False)
      .reset_index(drop=True)
)

display(comparison)

comparison.plot(
    x="Model",
    y=["Train F1", "Validation F1", "CV Mean F1"],
    kind="bar",
    figsize=(12, 5)
)
plt.title("Model Performance Comparison")
plt.ylabel("F1 score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

best_model_name = comparison.iloc[0]["Model"]
best_model = trained_models[best_model_name]
print("Provisional selected model:", best_model_name)

Selection should consider:

1. Validation F1.
2. Cross-validation mean and stability.
3. Overfitting gap.
4. Training and inference cost.
5. Explainability and operational simplicity.

The numerically highest score is not always the best production choice.

## 17. Validation diagnostics

In [ ]:
valid_pred = best_model.predict(X_valid)
valid_prob = best_model.predict_proba(X_valid)[:, 1]

print(classification_report(y_valid, valid_pred, digits=4))

ConfusionMatrixDisplay.from_predictions(y_valid, valid_pred)
plt.title(f"Confusion Matrix: {best_model_name}")
plt.show()

RocCurveDisplay.from_predictions(y_valid, valid_prob)
plt.title(f"ROC Curve: {best_model_name}")
plt.show()

PrecisionRecallDisplay.from_predictions(y_valid, valid_prob)
plt.title(f"Precision-Recall Curve: {best_model_name}")
plt.show()

## 18. Threshold tuning

In [ ]:
thresholds = np.arange(0.20, 0.81, 0.02)
threshold_rows = []

for threshold in thresholds:
    pred = (valid_prob >= threshold).astype(int)
    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_valid, pred, zero_division=0),
        "recall": recall_score(y_valid, pred, zero_division=0),
        "f1": f1_score(y_valid, pred, zero_division=0),
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df.sort_values("f1", ascending=False).head(10))

threshold_df.plot(x="threshold", y=["precision", "recall", "f1"])
plt.title("Validation Metrics at Different Thresholds")
plt.ylabel("Score")
plt.show()

best_threshold = threshold_df.loc[threshold_df["f1"].idxmax(), "threshold"]
print("Best validation threshold by F1:", round(float(best_threshold), 2))

Threshold tuning changes the decision policy, not the underlying probability model. It must be performed on validation data, never on the final test set.

## 19. Error analysis

In [ ]:
error_analysis = valid_df[
    ["date", "event_name", "ground_name", "team_1", "team_2",
     "toss_winner", "toss_decision", "winner", "team_1_win"]
].copy()

error_analysis["predicted_probability"] = valid_prob
error_analysis["prediction"] = (valid_prob >= best_threshold).astype(int)
error_analysis["correct"] = error_analysis["prediction"] == error_analysis["team_1_win"]
error_analysis["confidence"] = np.where(
    error_analysis["prediction"] == 1,
    error_analysis["predicted_probability"],
    1 - error_analysis["predicted_probability"]
)

mistakes = error_analysis[~error_analysis["correct"]].sort_values(
    "confidence", ascending=False
)

print("Validation errors:", len(mistakes))
display(mistakes.head(20))

Investigate confident mistakes. They often reveal:

- Missing explanatory variables.
- Rare teams or venues.
- Changes in team quality over time.
- Data-quality problems.
- Genuine randomness in sport.

## 20. Model interpretation with permutation importance

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_model,
    X_valid,
    y_valid,
    scoring="f1",
    n_repeats=10,
    random_state=SEED,
    n_jobs=-1,
)

importance_df = pd.DataFrame({
    "feature": X_valid.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

display(importance_df)

importance_df.sort_values("importance_mean").plot(
    x="feature", y="importance_mean", kind="barh", legend=False
)
plt.title("Permutation Importance on Validation Data")
plt.xlabel("Decrease in F1 after shuffling")
plt.show()

## 21. SHAP explanation

In [ ]:
import shap

# Use the transformed validation matrix and the fitted estimator.
fitted_preprocessor = best_model.named_steps["preprocessor"]
fitted_estimator = best_model.named_steps["model"]

X_valid_transformed = fitted_preprocessor.transform(X_valid)
sample = X_valid_transformed.sample(min(200, len(X_valid_transformed)), random_state=SEED)

try:
    explainer = shap.Explainer(fitted_estimator, sample)
    shap_values = explainer(sample)
    shap.plots.beeswarm(shap_values, max_display=15)

    # Explain one individual prediction.
    shap.plots.waterfall(shap_values[0], max_display=15)
except Exception as exc:
    print("Direct SHAP explanation was not supported for this selected estimator.")
    print("Use permutation importance above, or select the fitted XGBoost/Random Forest model.")
    print("Details:", str(exc)[:500])

SHAP values estimate how features move a prediction away from a reference prediction. They explain model behaviour, not causality. A feature can be predictive without causing the outcome.

## 22. Secondary regression task: predict Team 1 total runs

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

regression_features = classification_features.copy()
reg_target = "team_1_total_runs"

reg_train = train_df.dropna(subset=[reg_target]).copy()
reg_valid = valid_df.dropna(subset=[reg_target]).copy()

Xr_train = reg_train[regression_features]
yr_train = reg_train[reg_target]
Xr_valid = reg_valid[regression_features]
yr_valid = reg_valid[reg_target]

reg_models = {
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=300, max_depth=12, min_samples_leaf=3,
        n_jobs=-1, random_state=SEED
    ),
    "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=SEED),
}

reg_results = []
reg_pipelines = {}

for name, estimator in reg_models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator),
    ])

    start = time.perf_counter()
    pipeline.fit(Xr_train, yr_train)
    fit_time = time.perf_counter() - start

    train_pred = pipeline.predict(Xr_train)
    valid_pred_r = pipeline.predict(Xr_valid)

    reg_results.append({
        "Model": name,
        "Train RMSE": mean_squared_error(yr_train, train_pred) ** 0.5,
        "Validation RMSE": mean_squared_error(yr_valid, valid_pred_r) ** 0.5,
        "Validation MAE": mean_absolute_error(yr_valid, valid_pred_r),
        "Validation R2": r2_score(yr_valid, valid_pred_r),
        "Training Time (s)": fit_time,
    })
    reg_pipelines[name] = pipeline

reg_comparison = pd.DataFrame(reg_results).sort_values("Validation RMSE")
display(reg_comparison)

In [ ]:
best_reg_name = reg_comparison.iloc[0]["Model"]
best_reg_model = reg_pipelines[best_reg_name]
reg_pred = best_reg_model.predict(Xr_valid)

plt.scatter(yr_valid, reg_pred, alpha=0.5)
lims = [
    min(float(yr_valid.min()), float(reg_pred.min())),
    max(float(yr_valid.max()), float(reg_pred.max()))
]
plt.plot(lims, lims, linestyle="--")
plt.xlabel("Actual Team 1 Runs")
plt.ylabel("Predicted Team 1 Runs")
plt.title(f"Actual vs Predicted: {best_reg_name}")
plt.show()

residuals = yr_valid - reg_pred
plt.scatter(reg_pred, residuals, alpha=0.5)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Runs")
plt.ylabel("Residual")
plt.title("Regression Residual Plot")
plt.show()

### Classification versus regression

- Winner prediction is **classification** because the output is a category.
- Team score prediction is **regression** because the output is a continuous numerical value.

For the regression model:
- **MAE** gives the average absolute run error.
- **RMSE** penalizes large errors more strongly.
- **R²** measures explained variance relative to a mean-based benchmark.

## 23. Final training and untouched test evaluation

In [ ]:
# Combine training and validation data after model and threshold selection.
train_valid_df = pd.concat([train_df, valid_df], ignore_index=True)
X_train_valid = train_valid_df[classification_features]
y_train_valid = train_valid_df["team_1_win"]

final_model = trained_models[best_model_name]
final_model.fit(X_train_valid, y_train_valid)

test_prob = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= best_threshold).astype(int)

final_metrics = {
    "Selected model": best_model_name,
    "Threshold": float(best_threshold),
    "Test accuracy": accuracy_score(y_test, test_pred),
    "Test precision": precision_score(y_test, test_pred, zero_division=0),
    "Test recall": recall_score(y_test, test_pred, zero_division=0),
    "Test F1": f1_score(y_test, test_pred, zero_division=0),
    "Test ROC-AUC": roc_auc_score(y_test, test_prob),
}

display(pd.Series(final_metrics, name="value").to_frame())
print(classification_report(y_test, test_pred, digits=4))

ConfusionMatrixDisplay.from_predictions(y_test, test_pred)
plt.title("Final Test Confusion Matrix")
plt.show()

The test result is reported once, after all model and threshold decisions are complete. Repeatedly checking the test set turns it into another validation set and weakens the credibility of the final estimate.

## 24. Save, reload, and verify the model

In [ ]:
import joblib

artifact = {
    "model": final_model,
    "threshold": float(best_threshold),
    "features": classification_features,
    "target_definition": "1 if winner equals team_1, otherwise 0",
}

MODEL_PATH = "matchwise_winner_pipeline.joblib"
joblib.dump(artifact, MODEL_PATH)

reloaded = joblib.load(MODEL_PATH)
reloaded_prob = reloaded["model"].predict_proba(X_test.head(5))[:, 1]

print("Saved model:", MODEL_PATH)
print("Reload verification:", np.allclose(test_prob[:5], reloaded_prob))
files.download(MODEL_PATH)

## 25. Reusable inference function

In [ ]:
def predict_match(input_data, artifact=reloaded):
    required = artifact["features"]

    if isinstance(input_data, dict):
        input_df = pd.DataFrame([input_data])
    elif isinstance(input_data, pd.DataFrame):
        input_df = input_data.copy()
    else:
        raise TypeError("input_data must be a dictionary or pandas DataFrame.")

    missing = [col for col in required if col not in input_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    probabilities = artifact["model"].predict_proba(input_df[required])[:, 1]
    predictions = (probabilities >= artifact["threshold"]).astype(int)

    output = input_df.copy()
    output["team_1_win_probability"] = probabilities
    output["predicted_team_1_win"] = predictions
    return output

example_match = {
    "event_name": "Example Tournament",
    "ground_name": "Example Stadium",
    "ground_city": "Bengaluru",
    "team_1": str(df["team_1"].mode().iloc[0]),
    "team_2": str(df["team_2"].mode().iloc[0]),
    "toss_winner": str(df["team_1"].mode().iloc[0]),
    "toss_decision": str(df["toss_decision"].mode().iloc[0]),
    "year": 2026,
    "month": 7,
    "day_of_week": 0,
}

display(predict_match(example_match))

## 26. Batch inference from a new CSV

In [ ]:
# Upload a CSV containing all required feature columns.
new_files = files.upload()
new_csvs = [name for name in new_files if name.lower().endswith(".csv")]

if new_csvs:
    new_matches = pd.read_csv(new_csvs[0])
    predictions = predict_match(new_matches)
    display(predictions.head())

    output_path = "matchwise_predictions.csv"
    predictions.to_csv(output_path, index=False)
    files.download(output_path)

## 27. Guided student exercises

1. Replace the chronological split with a stratified random split. Compare the scores and explain why they differ.
2. Add `match_id` as a feature. Does the score improve? Is the improvement trustworthy?
3. Deliberately add `team_1_total_runs` and `team_2_total_runs` to winner prediction. Observe the apparent improvement and explain why it is leakage.
4. Tune Random Forest using `RandomizedSearchCV`.
5. Compare F1, ROC-AUC, and accuracy when changing the decision threshold.
6. Remove toss-related features and measure the change.
7. Compare one-hot encoding with frequency encoding.
8. Train XGBoost on CPU and GPU and compare training times.
9. Use SHAP to explain a confident correct prediction and a confident error.
10. Add recent team-form features computed only from earlier matches.

### Advanced challenge
Build expanding historical team-strength features:

- Matches played before the current date.
- Previous win rate.
- Average runs in the previous five matches.
- Venue-specific historical win rate.

Ensure every feature uses only information available before the match being predicted.

## 28. Knowledge check

1. Why is match winner a classification target?
2. Why is Team 1 total runs a regression target?
3. What is target leakage?
4. Why should preprocessing be inside a pipeline?
5. What does a large train–validation gap suggest?
6. How does regularizing a Decision Tree reduce overfitting?
7. Why use cross-validation?
8. What is the difference between GridSearchCV and RandomizedSearchCV?
9. How does Random Forest differ from boosting?
10. Why can accuracy be misleading?
11. What does threshold tuning change?
12. Why should the test set be evaluated only once?
13. What does permutation importance measure?
14. What do SHAP values explain?
15. Why does predictive importance not prove causality?

## 29. Common errors and troubleshooting

- **`KeyError` for a column:** Print `df.columns` and check spelling and whitespace.
- **Unknown category during inference:** Keep `handle_unknown="ignore"` in `OneHotEncoder`.
- **CUDA error in XGBoost:** Use `device="cpu"`; the rest of the notebook remains valid.
- **SHAP incompatibility:** Explain XGBoost or Random Forest directly, or use permutation importance.
- **Slow cross-validation:** Reduce `n_estimators`, tuning iterations, or folds during classroom demonstrations.
- **Very high validation score:** Audit for leakage before celebrating.
- **Poor future performance:** Prefer chronological validation and add time-aware historical features.
- **Missing input columns:** Use the exact feature schema stored with the model.

## 30. Conclusion and best practices

A reliable machine-learning workflow is not merely “fit and predict.” It requires:

1. A clearly defined prediction moment.
2. Leakage-safe features.
3. Realistic validation.
4. Suitable metrics.
5. Baseline comparison.
6. Overfitting diagnosis.
7. Disciplined tuning.
8. Explainability and error analysis.
9. Reproducible preprocessing.
10. Saved artifacts and validated inference.

The central lesson is simple:

> **A model is valuable not when it remembers historical matches, but when it generalizes responsibly to matches it has not seen.**